In [53]:
# Start with imports - ask the Cursor Agent to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import AsyncOpenAI
from IPython.display import Markdown, display

In [54]:
load_dotenv(override=True)

True

In [55]:
# For this homework assignment, we will be using ollama, the OpenAI API and Google API. Make sure to set your API keys in the .env file.
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key is set")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key is set")
else:
    print("Google API Key not set")

OpenAI API Key is set
Google API Key is set


In [56]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [57]:
messages

[{'role': 'user',
  'content': '\nPlease come up with a challenging, nuanced question with a succinct answer,\nthat I can ask a number of LLMs to evaluate their intelligence.\nNot a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.\nInclude in your question that the answer must be short.\nAnswer only with the question, no explanation.'}]

In [58]:
openai = AsyncOpenAI()

async def call_openai(client, model, messages):
    response = await client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

question = await call_openai(openai, model="gpt-5.4-mini", messages=messages)
display(Markdown(question))

You’re advising a city that wants to reduce traffic congestion without building new roads. Would you prioritize pricing, transit, zoning, or remote work incentives—and why? Answer in one sentence, choosing only the single most effective lever.

## Calling LLMs from mulitple providers

I will call multiple providers concurrently using `AsyncOpenAI`, so the
parallelization pattern is reflected not only conceptually but also in the
execution.

In [59]:
# OpenAI Compatible URLs

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [60]:
# OpenAI client libraries with the right base_url and key
# If this surprises you, please see Guide 9 in the Guides folder!

gemini = AsyncOpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)
ollama = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [61]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [62]:
def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

# Calling Ollama

My Ollama instance is running locally at `http://localhost:11434/`.

In [63]:
import requests
requests.get('http://localhost:11434').content

b'Ollama is running'

In [64]:
import requests
models = requests.get('http://localhost:11434/v1/models').json()
for model in models.get("data"):
    print(model.get("id"))

qwen3:1.7b
llama3.2:latest


## Parallel execution

The original lab uses multiple independent LLMs to answer the same question.
In this variation, I execute those independent calls concurrently using
`AsyncOpenAI` and `asyncio.gather()`.

This keeps the same parallelization pattern at the workflow level, while also
making the execution itself concurrent.

In [65]:
# The API we know well
# Reasoning effort can be none, low, medium, high, or xhigh
async def call_llm(client, model_name, messages):
  answer = await call_openai(client=client, model=model_name, messages=messages)
  return model_name, answer

openai_task = call_llm(
    openai,
    "gpt-5.4-mini",
    messages
)

openai_task_2 =  call_llm(
    openai,
    "gpt-5.4-nano",
    messages
)

gemini_task =  call_llm(
    gemini,
    "gemini-3.6-flash",
    messages
)

gemini_task_2 =  call_llm(
    gemini,
    "gemini-3.1-flash-lite",
    messages
)

ollama_task =  call_llm(
    ollama,
    "llama3.2:latest",
    messages
)

import asyncio

responses = await asyncio.gather(
    openai_task,
    gemini_task,
    ollama_task,
    openai_task_2,
    gemini_task_2
)

In [66]:
competitors = []
answers = []

for model_name, answer in responses:
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

I would prioritize **pricing**, because congestion is driven most directly by excess peak-hour driving, and road pricing or congestion charges change behavior immediately and broadly, reducing demand more efficiently than the slower, more indirect effects of transit, zoning, or remote-work incentives.

I would prioritize congestion pricing, as directly internalizing the cost of peak-hour driving is the most immediate, economically proven method to reduce demand and manage gridlock on existing infrastructure without waiting years for capital projects to build out.

I would prioritize zoning regulations as the most effective lever, as restricting development to high-density, pedestrian-friendly, and transit-oriented areas would reduce the number of cars on the road, decrease the demand for new roads, and promote a more compact, walkable, and livable urban environment.

Prioritize congestion pricing, because charging drivers based on road usage most directly and immediately reduces peak-hour demand without requiring new infrastructure.

I would prioritize **transit**—specifically high-frequency, high-capacity rail and bus rapid transit—because it is the only lever that simultaneously moves the greatest number of people in the smallest amount of space while creating the permanent density necessary to eliminate the need for car travel entirely.

In [67]:
print(f"Number of competitors: {len(competitors)}")

# Let's bring this together - note the use of "enumerate"
together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

Number of competitors: 5


In [68]:
print(together)

# Response from competitor 1

I would prioritize **pricing**, because congestion is driven most directly by excess peak-hour driving, and road pricing or congestion charges change behavior immediately and broadly, reducing demand more efficiently than the slower, more indirect effects of transit, zoning, or remote-work incentives.

# Response from competitor 2

I would prioritize congestion pricing, as directly internalizing the cost of peak-hour driving is the most immediate, economically proven method to reduce demand and manage gridlock on existing infrastructure without waiting years for capital projects to build out.

# Response from competitor 3

I would prioritize zoning regulations as the most effective lever, as restricting development to high-density, pedestrian-friendly, and transit-oriented areas would reduce the number of cars on the road, decrease the demand for new roads, and promote a more compact, walkable, and livable urban environment.

# Response from competitor 4



## Ranking the responses

Once all models have answered, a separate local model evaluates the responses
and ranks them from best to worst.

I use `qwen3:1.7b` through Ollama as the judge so that the ranking step is
independent from the models participating in the competition.

In [69]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [70]:
print(judge)

You are judging a competition between 5 competitors.
Each model has been given this question:

You’re advising a city that wants to reduce traffic congestion without building new roads. Would you prioritize pricing, transit, zoning, or remote work incentives—and why? Answer in one sentence, choosing only the single most effective lever.

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Here are the responses from each competitor:

# Response from competitor 1

I would prioritize **pricing**, because congestion is driven most directly by excess peak-hour driving, and road pricing or congestion charges change behavior immediately and broadly, reducing demand more efficiently than the slower, more indirect effects of transit, zoning, or remote-work ince

In [71]:
judge_messages = [{"role": "user", "content": judge}]

# I will use qwen3:1.7b locally 

Ollama highlights Qwen3 for its reasoning, instruction-following, and agentic capabilities; it is well-suited for evaluating responses against specific criteria.

In [72]:
model_name = "qwen3:1.7b"

response = await ollama.chat.completions.create(model=model_name, messages=judge_messages)
results = response.choices[0].message.content
print(results)

{"results": ["4", "5", "2", "1", "3"]}


In [73]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gpt-5.4-nano
Rank 2: gemini-3.1-flash-lite
Rank 3: gemini-3.6-flash
Rank 4: gpt-5.4-mini
Rank 5: llama3.2:latest


In [74]:
# Let's now use the winning model to optimize the answer
model_clients = {
    "gpt-5.4-mini": openai,
    "gpt-5.4-nano": openai,
    "gemini-3.6-flash": gemini,
    "gemini-3.1-flash-lite": gemini,
    "llama3.2:latest": ollama,
}

## Adding an Evaluator-Optimizer pattern

As an additional agentic pattern, I take the highest-ranked answer and improve
it iteratively.

The local Qwen model acts as the **Evaluator**. It decides whether the current
answer is sufficiently clear, accurate, complete, and well argued. If not, it
returns specific feedback.

The model that originally produced the winning answer acts as the
**Optimizer**, using that feedback to generate an improved answer.

In [75]:
# As an extra step, I will use the judge as the Evaluator
# and the winning model as the Optimizer.

async def evaluate(question, answer):
  evaluation_prompt = f"""
You are an evaluator of answers to questions.

    Question:
    {question}

    Answer:
    {answer}

    Evaluate whether the answer is sufficiently clear, accurate,
    complete, and well argued.

    Respond with JSON only, using exactly this format:
    {{"approved": true, "feedback": "feedback here"}}

    "approved" must be a JSON boolean.
    If the answer is already sufficiently good, set approved to true.
    Otherwise set it to false and provide specific feedback explaining
    what should be improved.

    Do not include markdown or code blocks."""

  messages=[{"role": "user", "content": evaluation_prompt}]
  response = await ollama.chat.completions.create(model=model_name, messages=messages)
  return json.loads(response.choices[0].message.content)
    

In [76]:
# optimization function that takes the winning model, the question, the current answer, and the feedback from the evaluation, and returns an optimized answer.
async def optimize(model_winner, question, answer, feedback):
    optimization_prompt = f"""You are an optimizer of answers to questions.
      You are given the following question:
      {question}
      You are also given the following answer:
      {answer}
      You are also given the following feedback:
      {feedback}
      Your job is to optimize the answer to be the best possible answer to the question, taking into account the feedback provided.
      Respond with the optimized answer only, no explanation or additional text."""

    messages=[{"role": "user", "content": optimization_prompt}]
    response = await model_clients[model_winner].chat.completions.create(model=model_winner, messages=messages)
    return response.choices[0].message.content

The evaluation-optimization loop is limited to a maximum of three evaluations.
The process stops early if the Evaluator approves the answer.

The iteration limit prevents an unbounded agentic loop and also keeps latency
and API usage under control.

In [77]:
# Here we will use the winning model to optimize the answer, using the evaluation and optimization functions defined above.
# The idea is improve the answer iteratively, until it is approved by the evaluator. In this way we use the pattern Evaluator-Optimizer after the parallel pattern of competition.
MAX_ITERATIONS = 3

best_answer_index = int(ranks[0]) - 1
model_winner = competitors[best_answer_index]
current_answer = answers[best_answer_index]

print(f"Winner model: {model_winner}")

for iteration in range(MAX_ITERATIONS):
    evaluation = await evaluate(question, current_answer)

    print(f"\nIteration {iteration + 1}")
    print(f"Approved: {evaluation['approved']}")
    print(f"Feedback: {evaluation['feedback']}")

    if evaluation["approved"]:
        break

    if iteration == MAX_ITERATIONS - 1:
        break

    current_answer = await optimize(
        model_winner,
        question,
        current_answer,
        evaluation["feedback"]
    )

display(Markdown(current_answer))

Winner model: gpt-5.4-nano



Iteration 1
Approved: True
Feedback: The answer is sufficiently clear, accurate, complete, and well-argued. It correctly identifies congestion pricing as the most effective lever for reducing traffic congestion without new infrastructure.


Prioritize congestion pricing, because charging drivers based on road usage most directly and immediately reduces peak-hour demand without requiring new infrastructure.

## Conclusion

This variation combines two agentic patterns:

1. **Parallelization** — multiple LLMs independently answer the same question,
   with the API calls executed concurrently.
2. **Evaluator-Optimizer** — the best answer is evaluated, improved using
   feedback, and evaluated again until it is approved or the iteration limit
   is reached.

The exercise also separates **roles** from **models**: the winning model becomes
the Optimizer, while Qwen serves as an independent Evaluator.